# 🏥 Healthcare Costs by State — Chest Pain Analysis (SOLUTION)

Complete expanded version with full EDA, multi-state boxplots, regional analysis, simulation, and clear takeaways.

Run all cells to generate plots and printed insights.


## 1. Theory: Using Boxplots for Multi-Group Comparison

Boxplots are particularly powerful when you want to compare the distribution of a numeric variable across many categories (in this case, U.S. states).

Key insights you can get from side-by-side boxplots:
- **Central tendency** (median line)
- **Spread** (height of the box = IQR)
- **Outliers** (individual points beyond whiskers)
- **Skewness** (position of the median within the box)

In healthcare cost analysis, this helps answer questions like:
- Which states have the highest/lowest charges for the same procedure?
- Which states have the most variation in charges?
- Are there systematic regional differences?


## 2. Loading the Data and Filtering for Chest Pain

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

# # Load the dataset (this file is large — be patient)
healthcare = pd.read_csv("/home/workdir/attachments/healthcare.csv")

print("Dataset shape:", healthcare.shape)
print("\nFirst 5 rows:")
print(healthcare.head())

# # Find the chest pain DRG
print("\nUnique DRGs (first 10):")
print(healthcare["DRG Definition"].unique()[:10])

# Filter for chest pain
chest_pain = healthcare[healthcare["DRG Definition"] == "313 - CHEST PAIN"]
print(f"\nNumber of chest pain records: {len(chest_pain)}")

## 3. Exploring One State (Example: Alabama)

In [ ]:
# # Filter for one state (e.g., Alabama)
state = "AL"
state_data = chest_pain[chest_pain["Provider State"] == state]

# # Get the costs (note the spaces in column name)
costs = state_data[" Average Covered Charges "].values

print(f"Alabama chest pain records: {len(costs)}")
print(f"Median charge in AL: ${np.median(costs):,.2f}")

# Boxplot for one state
plt.figure(figsize=(6, 5))
plt.boxplot(costs)
plt.title(f"Chest Pain Charges in {state}")
plt.ylabel("Average Covered Charges ($)")
plt.show()

## 4. Creating Boxplots for All States

In [ ]:
# # Get list of all states
states = chest_pain["Provider State"].unique()
print(f"Number of states: {len(states)}")

# Prepare data for boxplots
datasets = []
for state in states:
    state_costs = chest_pain[chest_pain["Provider State"] == state][" Average Covered Charges "].values
    datasets.append(state_costs)

# Create figure large enough for 50+ boxplots
plt.figure(figsize=(20, 8))
plt.boxplot(datasets, labels=states)
plt.title("Chest Pain Average Covered Charges by State", fontsize=16)
plt.xlabel("State")
plt.ylabel("Average Covered Charges ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Deeper Statistical Analysis by State

In [ ]:
def analyze_state(state, data):
    charges = data[" Average Covered Charges "]
    q1, median, q3 = np.percentile(charges, [25, 50, 75])
    iqr = q3 - q1
    outliers = charges[(charges < q1 - 1.5*iqr) | (charges > q3 + 1.5*iqr)]
    return {
        "State": state,
        "Count": len(charges),
        "Median": median,
        "IQR": iqr,
        "Outliers": len(outliers),
        "Min": charges.min(),
        "Max": charges.max()
    }

# Analyze all states
results = []
for state in states:
    state_df = chest_pain[chest_pain["Provider State"] == state]
    results.append(analyze_state(state, state_df))

results_df = pd.DataFrame(results).sort_values("Median", ascending=False)

print("Top 5 States by Median Chest Pain Charge:")
print(results_df.head().round(2).to_string(index=False))

print("\nBottom 5 States by Median Chest Pain Charge:")
print(results_df.tail().round(2).to_string(index=False))

## 6. Regional Analysis (Bonus)

In [ ]:
# Simple region mapping
regions = {
    "Northeast": ["ME","NH","VT","MA","RI","CT","NY","NJ","PA"],
    "Midwest": ["OH","IN","IL","MI","WI","MN","IA","MO","ND","SD","NE","KS"],
    "South": ["DE","MD","VA","WV","KY","NC","SC","TN","GA","FL","AL","MS","AR","LA","OK","TX"],
    "West": ["MT","ID","WY","CO","NM","AZ","UT","NV","CA","OR","WA","AK","HI"]
}

def get_region(state):
    for region, states_list in regions.items():
        if state in states_list:
            return region
    return "Other"

chest_pain["Region"] = chest_pain["Provider State"].apply(get_region)

plt.figure(figsize=(10, 6))
sns.boxplot(data=chest_pain, x="Region", y=" Average Covered Charges ", order=["Northeast","Midwest","South","West"])
plt.title("Chest Pain Charges by U.S. Region")
plt.ylabel("Average Covered Charges ($)")
plt.show()

print("Median charge by region:")
print(chest_pain.groupby("Region")[" Average Covered Charges "].median().sort_values(ascending=False).round(2))

## 7. 🎮 Simulation: Explore Different Scenarios

Modify the parameters below to analyze different diagnoses or payment types.


In [ ]:
# ============== SIMULATION PARAMETERS ==============
DRG_CODE = "313 - CHEST PAIN"           # Change to other DRG if desired
COLUMN_TO_PLOT = " Average Covered Charges "  # or "Average Medicare Payments"
# ========================================================

filtered = healthcare[healthcare["DRG Definition"] == DRG_CODE]
print(f"Records for {DRG_CODE}: {len(filtered)}")

# Re-run the all-states boxplot and stats with new parameters if desired
# (You can copy relevant code blocks and modify them)

## 8. Improved Visualizations with Seaborn

In [ ]:
# Seaborn boxplot (cleaner and more customizable)
plt.figure(figsize=(20, 8))
sns.boxplot(data=chest_pain, x="Provider State", y=" Average Covered Charges ", 
            order=sorted(chest_pain["Provider State"].unique()))
plt.title("Chest Pain Charges by State (Seaborn)", fontsize=16)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Violin plot for selected states (e.g., top 10 by median)
top_states = results_df.head(10)["State"].tolist()
subset = chest_pain[chest_pain["Provider State"].isin(top_states)]

plt.figure(figsize=(12, 6))
sns.violinplot(data=subset, x="Provider State", y=" Average Covered Charges ", 
               order=top_states, inner="quartile")
plt.title("Violin Plot of Top 10 States by Median Charge")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 🗺️ Analysis Flowchart

```mermaid
flowchart TD
    A[Load healthcare.csv] --> B[Filter for Chest Pain DRG]
    B --> C[Explore unique states and DRGs]
    C --> D[Create boxplots for all states]
    D --> E[Calculate median, IQR, outliers per state]
    E --> F[Regional grouping and comparison]
    F --> G[Simulation: Change DRG or column]
    G --> H[Advanced visualizations with seaborn]
    H --> I[Key Takeaways & Insights]
```


## 9. Practice Exercises & Key Takeaways

**Practice:**
1. Repeat the entire analysis for a different DRG (e.g., a common surgical procedure).
2. Compare `' Average Total Payments '` instead of Covered Charges.
3. Create a bar chart showing the number of chest pain discharges per state.
4. Identify states with the highest proportion of outliers.

**Key Takeaways from This Analysis:**
- There is **substantial variation** in hospital charges for the same diagnosis across states.
- Some states have much higher median charges and greater variability.
- Regional patterns exist but are not always straightforward (e.g., not simply "South is cheapest").
- Outliers are common — some hospitals charge dramatically more than others even within the same state.
- This highlights the complexity and lack of price transparency in the U.S. healthcare system.

This type of analysis is very relevant for health policy, insurance, and hospital administration work.
